## sankey 绘图

### 1. 基础示例1

In [1]:
import os
import numpy as np
import pandas as pd
from sankey import sankey, load_preset, from_wide, from_capacity

os.makedirs("./result/paper_figures", exist_ok=True)

# ═══════════════════════════════════════════════════════════
# 方式1: 长表格式 — 完全控制节点名和颜色
# ═══════════════════════════════════════════════════════════
nodes = pd.DataFrame({
    "name":  ["Amp", "Mut", "Del", "Path_A", "Path_B", "Path_C", "Cancer"],
    "layer": ["Inputs", "Inputs", "Inputs", "H1", "H1", "H1", "Outcome"],
    "is_residual": [False, False, False, False, False, False, False],
})
edges = pd.DataFrame({
    "source": [0, 0, 1, 1, 2, 2, 3, 4, 4, 5],
    "target": [3, 4, 4, 5, 5, 3, 6, 6, 6, 6],
    "value":  [30, 25, 15, 12, 18, 0, 55, 28, 12, 18],
})

# 自定义 Outcome: 更大的厚度、居中位置、重命名
fig1 = sankey(
    (edges, nodes),
    preset="nature",
    outcome_color="#5A1010",   # 深红突出显示
    height=500, width=1400,
)
fig1.write_html("./result/paper_figures/demo_outcome_long.html")

### 2. 基础示例2

In [ ]:
# ═══════════════════════════════════════════════════════════
# 方式2: 用 Sankey 风格 presets 对比
# ═══════════════════════════════════════════════════════════
df = pd.DataFrame({
    "Input":   ["Amp","Amp","Amp","Mut","Mut","Mut","Del","Del"],
    "Process": ["Immune","Metab","Signal","Immune","Metab","CellCycle","Metab","Signal"],
    "Outcome": ["Cancer","Cancer","Cancer","Cancer","Cancer","Cancer","Cancer","Cancer"],
})

for preset_name in ["nature", "cell", "science"]:
    fig = sankey(
        df, layer_cols=["Input", "Process", "Outcome"],
        preset=preset_name,
        height=450, width=1200,
        title=f"方式2: 宽表 — preset='{preset_name}'",
    )
    fig.write_html(f"./result/paper_figures/demo_outcome_{preset_name}.html")

### 3. 基础示例3

In [ ]:
# ═══════════════════════════════════════════════════════════
# 方式3: 容量字典 — Outcome 单节点自动生成
# ═══════════════════════════════════════════════════════════
caps = {
    "Inputs":  [550, 270, 180],
    "H1":      [400, 350, 250],
    "Outcome": [1000],
}

fig3 = sankey(
    caps,
    preset="nature",
    seed=0,
    height=450, width=1200,
    title="方式3: 容量字典 — Outcome 自动 Sinkhorn 分配",
)
fig3.write_html("./result/paper_figures/demo_outcome_capacity.html")

print("Done! 5 个 HTML 文件已保存到 result/paper_figures/")
print("  - demo_outcome_long.html      长表，Outcome 完全自定义")
print("  - demo_outcome_nature.html    Nature 预设")
print("  - demo_outcome_cell.html      Cell 预设")
print("  - demo_outcome_science.html   Science 预设")
print("  - demo_outcome_capacity.html  容量字典自动生成")

### 1. 文章复现Nature

In [ ]:
import os, numpy as np, pandas as pd
from sankey import sankey

os.makedirs("./result/paper_figures", exist_ok=True)

# ═══════════════════════════════════════════════════════════
# 1. Node definitions 
# ═══════════════════════════════════════════════════════════

INPUTS  = ["Amplification", "Mutation", "Deletion"]

H1 = ["AR", "TP53", "PTEN", "RB1", "MDM4",
      "FGFR1", "MAML3", "PDGFA", "NOTCH1", "EIF3E", "Residual"]

H2 = ["Ub-specific proc. proteases", "HSP90 SHR",
      "Neutrophil degranulation", "PKN1 and AR transcription",
      "SUMOylation", "NR transcription pathway",
      "Antigen processing", "RUNX2 and bone",
      "Regulation of TP53 Activity", "TP53 metabolic regulation",
      "Residual"]

H3 = ["SUMO E3 ligases", "TP53 transc. regulation",
      "RUNX2 transc. regulation", "G2/M transition",
      "RHO GTPases activate PKNs", "PTEN regulation",
      "Mitotic prophase", "Mitotic metaphase-anaphase",
      "Mitotic prometaphase", "Cap-dependent translation",
      "Residual"]

H4 = ["Generic transc. pathway", "Deubiquitination",
      "SUMOylation", "Rho GTPase effectors",
      "M phase", "Class I MHC pathway",
      "PIP3 activates AKT signalling", "Mitotic G2-G2/M phases",
      "Cellular senescence", "Eukaryotic translation",
      "Residual"]

H5 = ["Post-transl. modification", "RNA Pol II transc.",
      "Cellular responses to stress", "Cell cycle, mitotic",
      "Adaptive immune system", "Innate immune system",
      "Signalling by Rho GTPases", "Intracellular signalling",
      "Translation", "Immune cytokine sig.",
      "Residual"]

H6 = ["Metabolism of proteins", "Transcription (general)",
      "Immune system", "Signal transduction",
      "External stimuli response", "Cell cycle"]

OUTCOME = ["Outcome"]

# ═══════════════════════════════════════════════════════════
# 2. Sampling probabilities 
# ═══════════════════════════════════════════════════════════

PROBS = {
    "Inputs":  [0.50, 0.30, 0.20],
    "H1":      [0.14, 0.09, 0.08, 0.07, 0.06, 0.05, 0.04, 0.03, 0.02, 0.02, 0.38],
    "H2":      [0.10, 0.08, 0.08, 0.06, 0.06, 0.06, 0.04, 0.04, 0.04, 0.04, 0.40],
    "H3":      [0.12, 0.09, 0.08, 0.07, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.28],
    "H4":      [0.12, 0.09, 0.08, 0.07, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.28],
    "H5":      [0.12, 0.14, 0.10, 0.10, 0.10, 0.10, 0.10, 0.05, 0.05, 0.06, 0.08],
    "H6":      [0.20, 0.20, 0.20, 0.15, 0.15, 0.10],
    "Outcome": [0.5],
}

ALL_NODES = {
    "Inputs":  INPUTS,
    "H1":      H1,
    "H2":      H2,
    "H3":      H3,
    "H4":      H4,
    "H5":      H5,
    "H6":      H6,
    "Outcome": OUTCOME,
}

# ═══════════════════════════════════════════════════════════
# 3. Simulate data 
# ═══════════════════════════════════════════════════════════

def simulate_r_style(n: int = 100, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    data = {}
    for layer in ["Inputs", "H1", "H2", "H3", "H4", "H5", "H6", "Outcome"]:
        nodes = ALL_NODES[layer]
        probs = np.array(PROBS[layer])
        probs = probs / probs.sum()
        data[layer] = rng.choice(nodes, size=n, p=probs)
    return pd.DataFrame(data)


df = simulate_r_style(n=100, seed=42)
print(f"Simulated: {len(df)} rows x {len(df.columns)} columns")
for col in df.columns:
    counts = df[col].value_counts()
    print(f"  {col}: {dict(counts)}")

# ═══════════════════════════════════════════════════════════
# 4. 自主开发的sankey绘图包
# ═══════════════════════════════════════════════════════════

nature_custom = {
    "main_palette":      ["#7B1515", "#B54848", "#D4956A", "#8BBDD4", "#4A6FAF", "#C4A35A"],
    "gradient_method":   "sequential",
    "gradient_lighten":  0.6,
    "input_colors":      ["#D4956A", "#8BBDD4", "#4A6FAF"],
    "residual_color":    "#F0F0F0",
    "residual_link_alpha": 0.38,
    "outcome_color":     "#5A1010",
    "outcome_link_alpha": 0.35,
    "default_link_alpha": 0.18,
    "font_family":       "Arial",
    "font_size":         18,
    "node_thickness":    25,
    "node_pad":          80,
}

layer_cols = ["Inputs", "H1", "H2", "H3", "H4", "H5", "H6", "Outcome"]

fig = sankey(
    df,
    layer_cols=layer_cols,
    preset=nature_custom,
    y_method="fixed_gap",
    gap=0.012,
    height=750,
    width=2200,
)

# 输出路径
base_path = "./result/paper_figures/sankey_nature"
out_html  = f"{base_path}.html"
out_png   = f"{base_path}.png"
out_pdf   = f"{base_path}.pdf"

# 1. 保存交互式 HTML
fig.write_html(out_html)
print(f"\nSaved HTML: {out_html}")

# 2. 保存 PNG（scale 提高分辨率，论文推荐 scale=2~3）
fig.write_image(out_png, width=1800, height=800, scale=2)
print(f"Saved PNG: {out_png}")

# 3. 保存 PDF（矢量图，期刊/论文首选，无损缩放）
fig.write_image(out_pdf, width=1800, height=800)
print(f"Saved PDF: {out_pdf}")

Simulated: 100 rows x 8 columns
  Inputs: {'Amplification': 53, 'Mutation': 33, 'Deletion': 14}
  H1: {'Residual': 37, 'AR': 14, 'PTEN': 11, 'TP53': 9, 'FGFR1': 7, 'NOTCH1': 6, 'RB1': 6, 'MDM4': 4, 'MAML3': 3, 'PDGFA': 2, 'EIF3E': 1}
  H2: {'Residual': 38, 'HSP90 SHR': 16, 'Ub-specific proc. proteases': 11, 'Neutrophil degranulation': 8, 'NR transcription pathway': 6, 'TP53 metabolic regulation': 5, 'PKN1 and AR transcription': 5, 'RUNX2 and bone': 4, 'Regulation of TP53 Activity': 3, 'Antigen processing': 2, 'SUMOylation': 2}
  H3: {'Residual': 29, 'TP53 transc. regulation': 9, 'RHO GTPases activate PKNs': 9, 'PTEN regulation': 8, 'SUMO E3 ligases': 8, 'Mitotic prophase': 7, 'Mitotic prometaphase': 7, 'Mitotic metaphase-anaphase': 7, 'G2/M transition': 6, 'RUNX2 transc. regulation': 5, 'Cap-dependent translation': 5}
  H4: {'Residual': 25, 'Generic transc. pathway': 14, 'Cellular senescence': 13, 'Deubiquitination': 11, 'SUMOylation': 7, 'Eukaryotic translation': 6, 'Mitotic G2-G2/M p

### 2. 文章复现Cell

In [5]:
import os, numpy as np, pandas as pd
from sankey import sankey

os.makedirs("./result/paper_figures", exist_ok=True)

# ═══════════════════════════════════════════════════════════
# 1. Node definitions 
# ═══════════════════════════════════════════════════════════

INPUTS  = ["Amplification", "Mutation", "Deletion"]

H1 = ["AR", "TP53", "PTEN", "RB1", "MDM4",
      "FGFR1", "MAML3", "PDGFA", "NOTCH1", "EIF3E", "Residual"]

H2 = ["Ub-specific proc. proteases", "HSP90 SHR",
      "Neutrophil degranulation", "PKN1 and AR transcription",
      "SUMOylation", "NR transcription pathway",
      "Antigen processing", "RUNX2 and bone",
      "Regulation of TP53 Activity", "TP53 metabolic regulation",
      "Residual"]

H3 = ["SUMO E3 ligases", "TP53 transc. regulation",
      "RUNX2 transc. regulation", "G2/M transition",
      "RHO GTPases activate PKNs", "PTEN regulation",
      "Mitotic prophase", "Mitotic metaphase-anaphase",
      "Mitotic prometaphase", "Cap-dependent translation",
      "Residual"]

H4 = ["Generic transc. pathway", "Deubiquitination",
      "SUMOylation", "Rho GTPase effectors",
      "M phase", "Class I MHC pathway",
      "PIP3 activates AKT signalling", "Mitotic G2-G2/M phases",
      "Cellular senescence", "Eukaryotic translation",
      "Residual"]

H5 = ["Post-transl. modification", "RNA Pol II transc.",
      "Cellular responses to stress", "Cell cycle, mitotic",
      "Adaptive immune system", "Innate immune system",
      "Signalling by Rho GTPases", "Intracellular signalling",
      "Translation", "Immune cytokine sig.",
      "Residual"]

H6 = ["Metabolism of proteins", "Transcription (general)",
      "Immune system", "Signal transduction",
      "External stimuli response", "Cell cycle"]

OUTCOME = ["Outcome"]

# ═══════════════════════════════════════════════════════════
# 2. Sampling probabilities 
# ═══════════════════════════════════════════════════════════

PROBS = {
    "Inputs":  [0.50, 0.30, 0.20],
    "H1":      [0.14, 0.09, 0.08, 0.07, 0.06, 0.05, 0.04, 0.03, 0.02, 0.02, 0.38],
    "H2":      [0.10, 0.08, 0.08, 0.06, 0.06, 0.06, 0.04, 0.04, 0.04, 0.04, 0.40],
    "H3":      [0.12, 0.09, 0.08, 0.07, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.28],
    "H4":      [0.12, 0.09, 0.08, 0.07, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.28],
    "H5":      [0.12, 0.14, 0.10, 0.10, 0.10, 0.10, 0.10, 0.05, 0.05, 0.06, 0.08],
    "H6":      [0.20, 0.20, 0.20, 0.15, 0.15, 0.10],
    "Outcome": [0.5],
}

ALL_NODES = {
    "Inputs":  INPUTS,
    "H1":      H1,
    "H2":      H2,
    "H3":      H3,
    "H4":      H4,
    "H5":      H5,
    "H6":      H6,
    "Outcome": OUTCOME,
}

# ═══════════════════════════════════════════════════════════
# 3. Simulate data (independent sampling per layer = R logic)
# ═══════════════════════════════════════════════════════════

def simulate_r_style(n: int = 100, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    data = {}
    for layer in ["Inputs", "H1", "H2", "H3", "H4", "H5", "H6", "Outcome"]:
        nodes = ALL_NODES[layer]
        probs = np.array(PROBS[layer])
        probs = probs / probs.sum()
        data[layer] = rng.choice(nodes, size=n, p=probs)
    return pd.DataFrame(data)


df = simulate_r_style(n=100, seed=42)
print(f"Simulated: {len(df)} rows x {len(df.columns)} columns")
for col in df.columns:
    counts = df[col].value_counts()
    print(f"  {col}: {dict(counts)}")

# ═══════════════════════════════════════════════════════════
# 4. 自主开发的sankey绘图包
# ═══════════════════════════════════════════════════════════
cell_custom={
        "main_palette":        ["#1B6CA8", "#3DA5D9", "#73BFB8", "#9ED9CC", "#F4A261", "#E76F51"],
        "gradient_method":     "sequential",
        "gradient_lighten":    0.55,
        "input_colors":        ["#1B6CA8", "#73BFB8", "#E76F51"],
        "residual_color":      "#EAEAEA",
        "residual_link_alpha": 0.38,
        "residual_link_color": "#bfbfbf",
        "outcome_color":       "#0D4F8B",
        "outcome_link_alpha":  0.35,
        "default_link_alpha":  0.18,
        "font_family":         "Arial",
        "font_size":           18,
        "node_thickness":      25,
        "node_pad":            80,
    }

layer_cols = ["Inputs", "H1", "H2", "H3", "H4", "H5", "H6", "Outcome"]

fig = sankey(
    df,
    layer_cols=layer_cols,
    preset=cell_custom,
    y_method="fixed_gap",
    gap=0.012,
    height=750,
    width=2200,
)

# 输出路径
base_path = "./result/paper_figures/sankey_cell"
out_html  = f"{base_path}.html"
out_png   = f"{base_path}.png"
out_pdf   = f"{base_path}.pdf"

# 1. 保存交互式 HTML
fig.write_html(out_html)
print(f"\nSaved HTML: {out_html}")

# 2. 保存 PNG（scale 提高分辨率，论文推荐 scale=2~3）
fig.write_image(out_png, width=1800, height=800, scale=2)
print(f"Saved PNG: {out_png}")

# 3. 保存 PDF（矢量图，期刊/论文首选，无损缩放）
fig.write_image(out_pdf, width=1800, height=800)
print(f"Saved PDF: {out_pdf}")

Simulated: 100 rows x 8 columns
  Inputs: {'Amplification': 53, 'Mutation': 33, 'Deletion': 14}
  H1: {'Residual': 37, 'AR': 14, 'PTEN': 11, 'TP53': 9, 'FGFR1': 7, 'NOTCH1': 6, 'RB1': 6, 'MDM4': 4, 'MAML3': 3, 'PDGFA': 2, 'EIF3E': 1}
  H2: {'Residual': 38, 'HSP90 SHR': 16, 'Ub-specific proc. proteases': 11, 'Neutrophil degranulation': 8, 'NR transcription pathway': 6, 'TP53 metabolic regulation': 5, 'PKN1 and AR transcription': 5, 'RUNX2 and bone': 4, 'Regulation of TP53 Activity': 3, 'Antigen processing': 2, 'SUMOylation': 2}
  H3: {'Residual': 29, 'TP53 transc. regulation': 9, 'RHO GTPases activate PKNs': 9, 'PTEN regulation': 8, 'SUMO E3 ligases': 8, 'Mitotic prophase': 7, 'Mitotic prometaphase': 7, 'Mitotic metaphase-anaphase': 7, 'G2/M transition': 6, 'RUNX2 transc. regulation': 5, 'Cap-dependent translation': 5}
  H4: {'Residual': 25, 'Generic transc. pathway': 14, 'Cellular senescence': 13, 'Deubiquitination': 11, 'SUMOylation': 7, 'Eukaryotic translation': 6, 'Mitotic G2-G2/M p

### 3. 文章复现Science

In [6]:
import os, numpy as np, pandas as pd
from sankey import sankey

os.makedirs("./result/paper_figures", exist_ok=True)

# ═══════════════════════════════════════════════════════════
# 1. Node definitions 
# ═══════════════════════════════════════════════════════════

INPUTS  = ["Amplification", "Mutation", "Deletion"]

H1 = ["AR", "TP53", "PTEN", "RB1", "MDM4",
      "FGFR1", "MAML3", "PDGFA", "NOTCH1", "EIF3E", "Residual"]

H2 = ["Ub-specific proc. proteases", "HSP90 SHR",
      "Neutrophil degranulation", "PKN1 and AR transcription",
      "SUMOylation", "NR transcription pathway",
      "Antigen processing", "RUNX2 and bone",
      "Regulation of TP53 Activity", "TP53 metabolic regulation",
      "Residual"]

H3 = ["SUMO E3 ligases", "TP53 transc. regulation",
      "RUNX2 transc. regulation", "G2/M transition",
      "RHO GTPases activate PKNs", "PTEN regulation",
      "Mitotic prophase", "Mitotic metaphase-anaphase",
      "Mitotic prometaphase", "Cap-dependent translation",
      "Residual"]

H4 = ["Generic transc. pathway", "Deubiquitination",
      "SUMOylation", "Rho GTPase effectors",
      "M phase", "Class I MHC pathway",
      "PIP3 activates AKT signalling", "Mitotic G2-G2/M phases",
      "Cellular senescence", "Eukaryotic translation",
      "Residual"]

H5 = ["Post-transl. modification", "RNA Pol II transc.",
      "Cellular responses to stress", "Cell cycle, mitotic",
      "Adaptive immune system", "Innate immune system",
      "Signalling by Rho GTPases", "Intracellular signalling",
      "Translation", "Immune cytokine sig.",
      "Residual"]

H6 = ["Metabolism of proteins", "Transcription (general)",
      "Immune system", "Signal transduction",
      "External stimuli response", "Cell cycle"]

OUTCOME = ["Outcome"]

# ═══════════════════════════════════════════════════════════
# 2. Sampling probabilities 
# ═══════════════════════════════════════════════════════════

PROBS = {
    "Inputs":  [0.50, 0.30, 0.20],
    "H1":      [0.14, 0.09, 0.08, 0.07, 0.06, 0.05, 0.04, 0.03, 0.02, 0.02, 0.38],
    "H2":      [0.10, 0.08, 0.08, 0.06, 0.06, 0.06, 0.04, 0.04, 0.04, 0.04, 0.40],
    "H3":      [0.12, 0.09, 0.08, 0.07, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.28],
    "H4":      [0.12, 0.09, 0.08, 0.07, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.28],
    "H5":      [0.12, 0.14, 0.10, 0.10, 0.10, 0.10, 0.10, 0.05, 0.05, 0.06, 0.08],
    "H6":      [0.20, 0.20, 0.20, 0.15, 0.15, 0.10],
    "Outcome": [0.5],
}

ALL_NODES = {
    "Inputs":  INPUTS,
    "H1":      H1,
    "H2":      H2,
    "H3":      H3,
    "H4":      H4,
    "H5":      H5,
    "H6":      H6,
    "Outcome": OUTCOME,
}

# ═══════════════════════════════════════════════════════════
# 3. Simulate data (independent sampling per layer = R logic)
# ═══════════════════════════════════════════════════════════

def simulate_r_style(n: int = 100, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    data = {}
    for layer in ["Inputs", "H1", "H2", "H3", "H4", "H5", "H6", "Outcome"]:
        nodes = ALL_NODES[layer]
        probs = np.array(PROBS[layer])
        probs = probs / probs.sum()
        data[layer] = rng.choice(nodes, size=n, p=probs)
    return pd.DataFrame(data)


df = simulate_r_style(n=100, seed=42)
print(f"Simulated: {len(df)} rows x {len(df.columns)} columns")
for col in df.columns:
    counts = df[col].value_counts()
    print(f"  {col}: {dict(counts)}")

# ═══════════════════════════════════════════════════════════
# 4. 自主开发的sankey绘图包
# ═══════════════════════════════════════════════════════════
science_custom={
        "main_palette":        ["#D32F2F", "#1976D2", "#757575", "#FFA000", "#388E3C", "#7B1FA2"],
        "gradient_method":     "sequential",
        "gradient_lighten":    0.50,
        "input_colors":        ["#D32F2F", "#1976D2", "#FFA000"],
        "residual_color":      "#EEEEEE",
        "residual_link_alpha": 0.38,
        "residual_link_color": "#bfbfbf",
        "outcome_color":       "#212121",
        "outcome_link_alpha":  0.35,
        "default_link_alpha":  0.18,
        "font_family":         "Arial",
        "font_size":           18,
        "node_thickness":      25,
        "node_pad":            80,
    }

layer_cols = ["Inputs", "H1", "H2", "H3", "H4", "H5", "H6", "Outcome"]

fig = sankey(
    df,
    layer_cols=layer_cols,
    preset=science_custom,
    y_method="fixed_gap",
    gap=0.012,
    height=750,
    width=2200,
)

# 输出路径
base_path = "./result/paper_figures/sankey_science"
out_html  = f"{base_path}.html"
out_png   = f"{base_path}.png"
out_pdf   = f"{base_path}.pdf"

# 1. 保存交互式 HTML
fig.write_html(out_html)
print(f"\nSaved HTML: {out_html}")

# 2. 保存 PNG（scale 提高分辨率，论文推荐 scale=2~3）
fig.write_image(out_png, width=1800, height=800, scale=2)
print(f"Saved PNG: {out_png}")

# 3. 保存 PDF（矢量图，期刊/论文首选，无损缩放）
fig.write_image(out_pdf, width=1800, height=800)
print(f"Saved PDF: {out_pdf}")

Simulated: 100 rows x 8 columns
  Inputs: {'Amplification': 53, 'Mutation': 33, 'Deletion': 14}
  H1: {'Residual': 37, 'AR': 14, 'PTEN': 11, 'TP53': 9, 'FGFR1': 7, 'NOTCH1': 6, 'RB1': 6, 'MDM4': 4, 'MAML3': 3, 'PDGFA': 2, 'EIF3E': 1}
  H2: {'Residual': 38, 'HSP90 SHR': 16, 'Ub-specific proc. proteases': 11, 'Neutrophil degranulation': 8, 'NR transcription pathway': 6, 'TP53 metabolic regulation': 5, 'PKN1 and AR transcription': 5, 'RUNX2 and bone': 4, 'Regulation of TP53 Activity': 3, 'Antigen processing': 2, 'SUMOylation': 2}
  H3: {'Residual': 29, 'TP53 transc. regulation': 9, 'RHO GTPases activate PKNs': 9, 'PTEN regulation': 8, 'SUMO E3 ligases': 8, 'Mitotic prophase': 7, 'Mitotic prometaphase': 7, 'Mitotic metaphase-anaphase': 7, 'G2/M transition': 6, 'RUNX2 transc. regulation': 5, 'Cap-dependent translation': 5}
  H4: {'Residual': 25, 'Generic transc. pathway': 14, 'Cellular senescence': 13, 'Deubiquitination': 11, 'SUMOylation': 7, 'Eukaryotic translation': 6, 'Mitotic G2-G2/M p